In [1]:
import ee
import xarray
import rioxarray
import rasterio
from rasterio.transform import from_origin
import threading
import warnings
import dask.dataframe as dd
import httplib2
from dask.distributed import Client
import json
import os
import numpy as np

In [2]:
#http_transport = httplib2.Http(disable_ssl_certificate_validation=True)
#ee.Initialize(
#    opt_url='https://earthengine-highvolume.googleapis.com',
#    http_transport=http_transport, project='calfuels')
json_key = "test_key.json"
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

In [3]:
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [2]:
def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

In [4]:
initialize_dict = {
    'opt_url': 'https://earthengine-highvolume.googleapis.com',
    'project': 'calfuels'
}

In [5]:
json_key = 'test-key.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
initialize_dict = {
    'credentials': credentials,
    'opt_url': 'https://earthengine-highvolume.googleapis.com'
}

In [6]:
client = Client(n_workers=2, memory_limit="8GiB", threads_per_worker=1)
client.run(ee.Initialize, **initialize_dict)

r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\distributed\node.py:183: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 55896 instead
  warnings.warn(


{'tcp://127.0.0.1:55911': None, 'tcp://127.0.0.1:55912': None}

In [6]:
client = Client(n_workers=2, memory_limit="16GiB", threads_per_worker=2)
client.run(ee.Initialize, **initialize_dict)

{'tcp://127.0.0.1:64321': None, 'tcp://127.0.0.1:64324': None}

In [2]:
client = Client(n_workers=2, memory_limit="8GiB", threads_per_worker=1)

In [4]:
client.run(ee.Initialize)

{'tcp://127.0.0.1:49953': None, 'tcp://127.0.0.1:49956': None}

In [11]:
client.close()

In [3]:
PREFERRED_CHUNKS = {
    'time': 48,
    'X': 512,
    'Y': 256
}

In [6]:
FAILING_CHUNKS = {
    'time': 1024,
    'X': 1024,
    'Y': 1024
}

In [6]:
CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterDate('2000-01-01', '2022-12-31').filterBounds(LTBMU.geometry())
first_img = ic.first()
ic_first = ee.ImageCollection(first_img)
ic_xr = xarray.open_dataset(ic, engine = "ee", chunks=PREFERRED_CHUNKS, crs='EPSG:3310', geometry=LTBMU.geometry(), scale = 100)

In [18]:
print(ic_xr)

<xarray.Dataset>
Dimensions:        (time: 218, X: 325, Y: 653)
Coordinates:
  * time           (time) datetime64[ns] 2013-03-22T18:36:56.975000 ... 2013-...
  * X              (X) float64 -2.187e+04 -2.177e+04 ... 1.043e+04 1.053e+04
  * Y              (Y) float64 7.654e+04 7.664e+04 ... 1.416e+05 1.417e+05
Data variables: (12/19)
    SR_B1          (time, X, Y) float32 ...
    SR_B2          (time, X, Y) float32 ...
    SR_B3          (time, X, Y) float32 ...
    SR_B4          (time, X, Y) float32 ...
    SR_B5          (time, X, Y) float32 ...
    SR_B6          (time, X, Y) float32 ...
    ...             ...
    ST_EMSD        (time, X, Y) float32 ...
    ST_QA          (time, X, Y) float32 ...
    ST_TRAD        (time, X, Y) float32 ...
    ST_URAD        (time, X, Y) float32 ...
    QA_PIXEL       (time, X, Y) float32 ...
    QA_RADSAT      (time, X, Y) float32 ...
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This

In [9]:
new_ic_xr = ic_xr.to_dask_dataframe()

In [10]:
print(new_ic_xr)

Dask DataFrame Structure:
                          time        X        Y    SR_B1    SR_B2    SR_B3    SR_B4    SR_B5    SR_B6    SR_B7 SR_QA_AEROSOL   ST_B10 ST_ATRAN ST_CDIST  ST_DRAD  ST_EMIS  ST_EMSD    ST_QA  ST_TRAD  ST_URAD QA_PIXEL QA_RADSAT
npartitions=15                                                                                                                                                                                                                   
0               datetime64[ns]  float64  float64  float32  float32  float32  float32  float32  float32  float32       float32  float32  float32  float32  float32  float32  float32  float32  float32  float32  float32   float32
3395600                    ...      ...      ...      ...      ...      ...      ...      ...      ...      ...           ...      ...      ...      ...      ...      ...      ...      ...      ...      ...      ...       ...
...                        ...      ...      ...      ...      ...    

In [7]:
CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

dem_img = ee.Image("USGS/3DEP/10m")
ic_first = ee.ImageCollection(dem_img)
ic_xr = xarray.open_dataset(ic_first, engine = "ee", crs='EPSG:3310', geometry=LTBMU.geometry(), scale = 10)

c:\Users\knigh\anaconda3\envs\big-raster\lib\site-packages\xee\ext.py:683: UserWarning: Unable to retrieve 'system:time_start' values from an ImageCollection due to: No 'system:time_start' values found in the 'ImageCollection'.
  warnings.warn(


In [18]:
ndvi = (ic_xr['SR_B5'] - ic_xr['SR_B4']) / (ic_xr['SR_B5'] + ic_xr['SR_B4'])
ic_xr['NDVI'] = ndvi

In [19]:
ic_xr_result = ic_xr.compute()

KeyboardInterrupt: 

In [51]:
print(ic_xr_result)

<xarray.Dataset>
Dimensions:        (time: 22, X: 1084, Y: 2178)
Coordinates:
  * time           (time) datetime64[ns] 2019-01-04T18:39:22.062000 ... 2019-...
  * X              (X) float64 -2.19e+04 -2.187e+04 ... 1.056e+04 1.059e+04
  * Y              (Y) float64 7.65e+04 7.653e+04 ... 1.418e+05 1.418e+05
Data variables: (12/20)
    SR_B1          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B2          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B3          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B4          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B5          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    SR_B6          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    ...             ...
    ST_QA          (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
    ST_TRAD        (time, X, Y) float32 nan nan nan nan nan ... nan nan nan nan
   

In [52]:
ic_xr_result = ic_xr_result.rename({'X': 'x', 'Y': 'y'})
ic_xr_result = ic_xr_result.transpose('time', 'y', 'x')

In [54]:
%%time
ic_xr_result["NDVI"].isel(time=0).rio.to_raster("ndvi.tif")

CPU times: total: 62.5 ms
Wall time: 48.8 ms


In [ ]:
# Lazy operation! Does not load the raster into memory!!
da = xarray.open_dataset("https://github.com/mapbox/rasterio/raw/1.2.1/tests/data/RGB.byte.tif")

In [ ]:
warnings.filterwarnings("ignore")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU")

#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'], 
          #['UBlue', 'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']).filterBounds(LTBMU.geometry())

ic_landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterBounds(LTBMU.geometry())
ic_dem = ee.ImageCollection("USGS/3DEP/1m").filterBounds(LTBMU.geometry())
ic_xr = xarray.open_mfdataset([ic_landsat, ic_dem], engine = "ee", crs='EPSG:3310', scale = 30)



In [ ]:
ds = xarray.open_mfdataset(['ee://ECMWF/ERA5_LAND/HOURLY', 'ee://NASA/GDDP-CMIP6'],
                           engine='ee', crs='EPSG:4326', scale=0.25)

In [5]:
red_band = ic_xr['SR_B4'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
nir_band = ic_xr['SR_B5'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
ndvi = (nir_band - red_band) / (nir_band + red_band)
ndvi_transpose = ndvi.transpose('Y', 'X').rename({"X": "x", "Y": "y"})

In [30]:
ic = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate('1992-10-05', '1993-03-31')
leg1 = ee.Geometry.Rectangle(113.33, -43.63, 153.56, -10.66)
ds = xarray.open_dataset(
    ic,
    engine='ee',
    projection=ic.first().select(0).projection(),
    geometry=leg1
)

In [ ]:
ds_renamed = ds.rename({'lon': 'x', 'lat': 'y'})
ds_renamed_transpose = ds_renamed.transpose('time', 'y', 'x')
ds_renamed_transpose.isel(time=0).rio.to_raster("hourly.tif")
#ds_renamed_transpose.rio.to_raster(output_raster_path, driver='GTiff', crs='EPSG:4326', compress='lzw')

In [ ]:
###############################
### TESTING MY PACKAGE CODE ###
###############################

In [14]:
print(os.getcwd())

r:\SCRATCH\adrianom\code\big-raster-helper\experiment_code


In [4]:
import sys
import os
import ee

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from initialize_dask import DaskHandler
from input_driver import EarthEngineReader

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

chunk_size  = {
            'time': 48,
            'X': 512,
            'Y': 256
        }

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'start_date': '2000-01-01',
            'end_date': '2022-12-31',
            'geometry': LTBMU.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'chunks': chunk_size,
            'map_function': prep_sr_l8
        }
landsat_getter = EarthEngineReader(parameters)
print(landsat_getter.dataset)

APPLELLELLE
None
<xarray.Dataset>
Dimensions:        (time: 218, X: 325, Y: 653)
Coordinates:
  * time           (time) datetime64[ns] 2013-03-22T18:36:56.975000 ... 2013-...
  * X              (X) float64 -2.187e+04 -2.177e+04 ... 1.043e+04 1.053e+04
  * Y              (Y) float64 7.654e+04 7.664e+04 ... 1.416e+05 1.417e+05
Data variables: (12/19)
    SR_B1          (time, X, Y) float32 dask.array<chunksize=(48, 325, 256), meta=np.ndarray>
    SR_B2          (time, X, Y) float32 dask.array<chunksize=(48, 325, 256), meta=np.ndarray>
    SR_B3          (time, X, Y) float32 dask.array<chunksize=(48, 325, 256), meta=np.ndarray>
    SR_B4          (time, X, Y) float32 dask.array<chunksize=(48, 325, 256), meta=np.ndarray>
    SR_B5          (time, X, Y) float32 dask.array<chunksize=(48, 325, 256), meta=np.ndarray>
    SR_B6          (time, X, Y) float32 dask.array<chunksize=(48, 325, 256), meta=np.ndarray>
    ...             ...
    ST_EMSD        (time, X, Y) float32 dask.array<chunksize=

In [9]:
print(dataset)

<xarray.Dataset>
Dimensions:        (time: 218, X: 325, Y: 653)
Coordinates:
  * time           (time) datetime64[ns] 2013-03-22T18:36:56.975000 ... 2013-...
  * X              (X) float64 -2.187e+04 -2.177e+04 ... 1.043e+04 1.053e+04
  * Y              (Y) float64 7.654e+04 7.664e+04 ... 1.416e+05 1.417e+05
Data variables: (12/19)
    SR_B1          (time, X, Y) float32 ...
    SR_B2          (time, X, Y) float32 ...
    SR_B3          (time, X, Y) float32 ...
    SR_B4          (time, X, Y) float32 ...
    SR_B5          (time, X, Y) float32 ...
    SR_B6          (time, X, Y) float32 ...
    ...             ...
    ST_EMSD        (time, X, Y) float32 ...
    ST_QA          (time, X, Y) float32 ...
    ST_TRAD        (time, X, Y) float32 ...
    ST_URAD        (time, X, Y) float32 ...
    QA_PIXEL       (time, X, Y) float32 ...
    QA_RADSAT      (time, X, Y) float32 ...
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This

In [5]:
import sys
import os
import ee

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from initialize_dask import DaskHandler
from input_driver import EarthEngineReader

handler = DaskHandler()
handler.create_local_cluster(n_workers=4, threads_per_worker=1, memory_limit="16GB")
# Maybe a method to close the cluster when done?

2024-06-28 15:33:26,756 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\tornado\websocket.py", line 954, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\tornado\web.py", line 3173, in wrapper
    return method(self, *args, **kwargs)
  File "r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\bokeh\server\views\ws.py", line 149, in open
    raise ProtocolError("Token is expired.")
bokeh.protocol.exceptions.ProtocolError: Token is expired.
Task exception was never retrieved
future: <Task finished name='Task-3285506' coro=<Client._gather.<locals>.

In [10]:
handler.initialize_earth_engine_dask_workers()
# If the crs is 4326 or anything geographic, I need a check to ensure that lon/lat are
# the dimension names!

In [11]:
dask_df = handler.dataset_to_dask_dataframe(chunked_dataset)

In [19]:
handler.dask_client.close()

In [7]:
chunks  = {
            'time': 48,
            'X': 512,
            'Y': 256
        }
handler.initialize_earth_engine_dask_workers()
chunked_dataset = handler.process_with_dask(dataset, chunks=chunks)
# If the crs is 4326 or anything geographic, I need a check to ensure that lon/lat are
# the dimension names!

In [14]:
def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

In [18]:
from udf_tuner import UserDefinedFunctionTuner
tuner = UserDefinedFunctionTuner(dask_df)
tuner.apply_function(compute_ndvi)

KeyboardInterrupt: 

In [9]:
ndvi = (chunked_dataset['SR_B5'] - chunked_dataset['SR_B4']) / (chunked_dataset['SR_B5'] + chunked_dataset['SR_B4'])
chunked_dataset['NDVI'] = ndvi

In [10]:
chunked_dataset.compute()

<xarray.Dataset>
Dimensions:        (time: 218, X: 325, Y: 653)
Coordinates:
  * time           (time) datetime64[ns] 2013-03-22T18:36:56.975000 ... 2013-...
  * X              (X) float64 -2.187e+04 -2.177e+04 ... 1.043e+04 1.053e+04
  * Y              (Y) float64 7.654e+04 7.664e+04 ... 1.416e+05 1.417e+05
Data variables: (12/20)
    SR_B1          (time, X, Y) float32 0.0153 0.00284 0.007597 ... nan nan nan
    SR_B2          (time, X, Y) float32 0.02699 0.02102 0.0186 ... nan nan nan
    SR_B3          (time, X, Y) float32 0.05066 0.05316 0.04703 ... nan nan nan
    SR_B4          (time, X, Y) float32 0.04236 0.0489 0.0387 ... nan nan nan
    SR_B5          (time, X, Y) float32 0.2297 0.229 0.2358 ... nan nan nan
    SR_B6          (time, X, Y) float32 0.04962 0.05066 0.07297 ... nan nan nan
    ...             ...
    ST_QA          (time, X, Y) float32 381.0 378.0 380.0 381.0 ... nan nan nan
    ST_TRAD        (time, X, Y) float32 6.993e+03 7.091e+03 ... nan nan
    ST_URAD        (time, X, Y) float32 128.0 128.0 128.0 126.0 ... nan nan nan
    QA_PIXEL       (time, X, Y) float32 2.182e+04 2.182e+04 ... nan nan
    QA_RADSAT      (time, X, Y) float32 0.0 0.0 0.0 0.0 0.0 ... nan nan nan nan
    NDVI           (time, X, Y) float32 0.6887 0.6481 0.718 ... nan nan nan
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:  SR_B7,SR_B5,SR_B3
    visualization_2_max:    30000.0
    visualization_2_min:    0.0
    visualization_2_name:   Shortwave Infrared (753)
    crs:                    EPSG:3310

In [ ]:
####################
#### UDF TUNING ####
####################

# Taking in a dask data frame
# Run a custom function on the data frame
# 
# Example usage:
if __name__ == "__main__":
    # Create a sample Dask DataFrame
    df = dd.from_pandas(pd.DataFrame({'x': range(10), 'y': range(10, 20)}), npartitions=2)
    
    # Define a sample user-defined function
    def sample_function(df):
        return df['x'] + df['y']
    
    # Initialize the DaskFunctionRunner with the Dask DataFrame
    runner = UserDefinedFunctionTuner(df)
    
    # Apply the user-defined function to the Dask DataFrame
    result = runner.apply_function(sample_function)
    print(result)

In [4]:
data1 = xarray.DataArray(np.random.rand(3, 4), dims=["x", "y"])
data2 = xarray.DataArray(np.random.rand(4), dims=["y"])

In [6]:
print(data2)
print(data1)

<xarray.DataArray (y: 4)>
array([0.8734092 , 0.41357864, 0.60003947, 0.35881585])
Dimensions without coordinates: y
<xarray.DataArray (x: 3, y: 4)>
array([[0.07700087, 0.98106335, 0.9402194 , 0.57474907],
       [0.05425729, 0.10617428, 0.02739126, 0.62920134],
       [0.34230875, 0.09563789, 0.79681057, 0.46266657]])
Dimensions without coordinates: x, y
